# Marathi QLoRA Fine-Tune — Qwen2.5-1.5B-Instruct (Unsloth path)

Author: Tushar Islampure (github.com/tusharislampure29)

Loads data from `tusharislampure29/marathi-instruct-30k` on HF Hub and pushes
the trained adapter to `tusharislampure29/qwen2.5-1.5b-marathi-instruct`.

**Why Unsloth here**: 2-5× T4 speedup, native fp16 handling that
avoids the `_amp_foreach_non_finite_check_and_unscale` bf16 crash we hit
with stock TRL on Qwen2.5's bf16 weights. Plus, gradient checkpointing
without the 30% slowdown.

**Resume-safe**: a `TrainerCallback` pushes the adapter to HF Hub after
every `save_steps`, so a Colab disconnect at hour 10 still leaves us with
the latest checkpoint on the Hub.

Before Run all: Runtime > Change runtime type > T4 GPU. Add `HF_TOKEN` (write) and `WANDB_API_KEY` to Colab Secrets (key icon, left sidebar).

In [ ]:
# 1. install Unsloth (handles torch / trl / peft / bnb version alignment)
%%capture
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q wandb

In [ ]:
# 2. auth + config
import os, torch
from google.colab import userdata

os.environ['HF_TOKEN']      = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

cfg = {
    'base_id':         'Qwen/Qwen2.5-1.5B-Instruct',
    'max_seq_length':  1024,
    'dataset':         'tusharislampure29/marathi-instruct-30k',
    'repo_id':         'tusharislampure29/qwen2.5-1.5b-marathi-instruct',

    'lora_r':           16,
    'lora_alpha':       32,
    'lora_dropout':     0.05,
    'target_modules':   ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],

    'num_epochs':       3,
    'batch_size':       4,
    'grad_accum':       8,
    'learning_rate':    2.0e-4,
    'lr_scheduler':     'cosine',
    'warmup_ratio':     0.03,
    'weight_decay':     0.0,
    'logging_steps':    25,
    'eval_steps':       200,
    'save_steps':       200,
    'save_total_limit': 3,
    'output_dir':       '/content/out',
}

print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
print('base:', cfg['base_id'])

In [ ]:
# 3. load base model in 4-bit via Unsloth FastLanguageModel
#    dtype=None lets Unsloth pick fp16 on Turing (T4) and bf16 on Ampere+.
#
#    Workaround: Unsloth's `time_limited_stats_check` hits a HF Hub stats
#    endpoint that was hanging for >120s on Colab today. Patch it to a no-op
#    so the model load uses the normal HF Hub download path.
import importlib
_uu = importlib.import_module('unsloth.models._utils')
_uu.time_limited_stats_check = lambda *a, **kw: None
_uu._get_statistics          = lambda *a, **kw: None

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = cfg['base_id'],
    max_seq_length = cfg['max_seq_length'],
    dtype          = None,
    load_in_4bit   = True,
    token          = os.environ['HF_TOKEN'],
)


In [ ]:
# 4. LoRA wrap via Unsloth (with its no-slowdown gradient checkpointing)
model = FastLanguageModel.get_peft_model(
    model,
    r              = cfg['lora_r'],
    lora_alpha     = cfg['lora_alpha'],
    lora_dropout   = cfg['lora_dropout'],
    target_modules = cfg['target_modules'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

In [ ]:
# 5. data — pulled from HF Hub (private dataset)
from datasets import load_dataset
train_ds = load_dataset(cfg['dataset'], data_files='train.jsonl', split='train', token=os.environ['HF_TOKEN'])
val_ds   = load_dataset(cfg['dataset'], data_files='val.jsonl',   split='train', token=os.environ['HF_TOKEN'])
print(f'train: {len(train_ds)}  val: {len(val_ds)}')
print('sample:\n', train_ds[0]['text'][:600])

In [ ]:
# 6. SFT training with resume-safe mid-run Hub pushes
#    - Saves every save_steps locally AND pushes that checkpoint to HF Hub.
#    - Auto-resumes from the latest HF-Hub checkpoint if one exists, so a Colab
#      session disconnect at hour N restarts at step N, not step 0.
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback
from huggingface_hub import HfApi, snapshot_download
import wandb

# Auto-resume: pull the latest checkpoint from HF Hub if there is one.
def find_resume_checkpoint(repo_id, hub_token, output_dir):
    api = HfApi(token=hub_token)
    try:
        files = api.list_repo_files(repo_id=repo_id, repo_type='model')
    except Exception as e:
        print(f'[resume] could not list repo ({e}); starting fresh')
        return None
    ckpts = sorted(
        {f.split('/')[0] for f in files if f.startswith('checkpoint-')},
        key=lambda s: int(s.split('-')[1]),
    )
    if not ckpts:
        print('[resume] no prior checkpoints on Hub; starting fresh')
        return None
    latest = ckpts[-1]
    os.makedirs(output_dir, exist_ok=True)
    print(f'[resume] downloading {latest} from {repo_id} ...')
    snapshot_download(
        repo_id=repo_id, repo_type='model',
        allow_patterns=[f'{latest}/*'],
        local_dir=output_dir,
        token=hub_token,
    )
    local = os.path.join(output_dir, latest)
    print(f'[resume] will resume from {local}')
    return local

resume_from = find_resume_checkpoint(cfg['repo_id'], os.environ['HF_TOKEN'], cfg['output_dir'])

# wandb: continue the same run id if we have one written to disk from a prior segment
_wandb_run_id_file = os.path.join(cfg['output_dir'], '.wandb_run_id')
_existing_run_id = open(_wandb_run_id_file).read().strip() if os.path.isfile(_wandb_run_id_file) else None
wandb.init(
    project='marathi-instruct-llm',
    name='qwen2.5-1.5b-qlora-v2-unsloth',
    id=_existing_run_id,
    resume='allow' if _existing_run_id else None,
)
os.makedirs(cfg['output_dir'], exist_ok=True)
open(_wandb_run_id_file, 'w').write(wandb.run.id)

class HubAdapterPushCallback(TrainerCallback):
    def __init__(self, repo_id, hub_token, output_dir):
        self.repo_id = repo_id
        self.output_dir = output_dir
        self.api = HfApi(token=hub_token)

    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = os.path.join(self.output_dir, f'checkpoint-{state.global_step}')
        if not os.path.isdir(ckpt_dir):
            return
        try:
            last_loss = state.log_history[-1].get('loss', 'n/a') if state.log_history else 'n/a'
            self.api.upload_folder(
                folder_path   = ckpt_dir,
                repo_id       = self.repo_id,
                repo_type     = 'model',
                path_in_repo  = f'checkpoint-{state.global_step}',
                commit_message= f'mid-training checkpoint-{state.global_step} (loss={last_loss})',
            )
            print(f'[hub] pushed checkpoint-{state.global_step}')
        except Exception as e:
            print(f'[hub] checkpoint-{state.global_step} push failed: {e}')

args = SFTConfig(
    output_dir                  = cfg['output_dir'],
    num_train_epochs            = cfg['num_epochs'],
    per_device_train_batch_size = cfg['batch_size'],
    gradient_accumulation_steps = cfg['grad_accum'],
    learning_rate               = cfg['learning_rate'],
    lr_scheduler_type           = cfg['lr_scheduler'],
    warmup_ratio                = cfg['warmup_ratio'],
    weight_decay                = cfg['weight_decay'],
    max_length                  = cfg['max_seq_length'],
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = cfg['logging_steps'],
    eval_strategy               = 'steps',
    eval_steps                  = cfg['eval_steps'],
    save_strategy               = 'steps',
    save_steps                  = cfg['save_steps'],
    save_total_limit            = cfg['save_total_limit'],
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'wandb',
    dataset_text_field          = 'text',
    seed                        = 42,
)
trainer = SFTTrainer(
    model            = model,
    args             = args,
    train_dataset    = train_ds,
    eval_dataset     = val_ds,
    processing_class = tokenizer,
    callbacks        = [HubAdapterPushCallback(cfg['repo_id'], os.environ['HF_TOKEN'], cfg['output_dir'])],
)
trainer.train(resume_from_checkpoint=resume_from)


In [ ]:
# 7. push best adapter to HF Hub (best == lowest eval_loss, from load_best_model_at_end)
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
model.push_to_hub(cfg['repo_id'])
tokenizer.push_to_hub(cfg['repo_id'])
print(f'pushed best adapter to https://huggingface.co/{cfg["repo_id"]}')

In [ ]:
# 8. sanity check — 3 Marathi prompts through the tuned model
FastLanguageModel.for_inference(model)
prompts = [
    'महाराष्ट्राची राजधानी कोणती आहे आणि तिथली लोकसंख्या किती आहे?',
    'महात्मा गांधींबद्दल तीन वाक्यांत माहिती द्या.',
    'एक चांगला सॉफ्टवेअर इंजिनिअर होण्यासाठी कोणते गुण आवश्यक आहेत?',
]
for p in prompts:
    msgs = [
        {'role': 'system', 'content': 'तुम्ही एक उपयुक्त AI सहाय्यक आहात जो मराठीत स्पष्ट आणि अचूक उत्तरे देतो.'},
        {'role': 'user',   'content': p},
    ]
    text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    print('Q:', p)
    print('A:', tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip())
    print('---')